# Baseline: Predict `failure_mode` from First 3 Iterations

LoopNet seed v0.1 — supervised learning on early trajectory windows.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from scripts.baseline_failure_mode import load_records, extract_features
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

records = load_records(ROOT / "data" / "seed" / "records.jsonl")
labeled = [r for r in records if r.get("failure_mode")]
train = [r for r in labeled if r["split"] == "train"]
test = [r for r in labeled if r["split"] == "test"]

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=200, random_state=42)),
])
model.fit([extract_features(r) for r in train], [r["failure_mode"] for r in train])
pred = model.predict([extract_features(r) for r in test])
print(classification_report([r["failure_mode"] for r in test], pred, zero_division=0))